# 04 Train SFT (A1, A2)

Trains a LoRA adapter on Tiny Aya for arm A1 (raw rows) and arm A2 (curated rows), three seeds each, with
AlignTune SFT and the LoRA settings in `configs/lora.yaml`. Input: `data/arms/{A1,A2}/train.jsonl`. Output:
`checkpoints/{arm}-s{seed}-{git_sha}/` with a `dhanground.json` record, optionally pushed to the Hub.
Manifest: `dhanground.json` per adapter (arm, seed, base model, backend); checkpoints sync to the Hub during
training so a dead GPU session can resume.

**Needs:** a GPU (L4), `pip install -r requirements-gpu.txt`, and `hf auth login` (Tiny Aya is gated).
If AlignTune fails on the chat template, set `BACKEND = "trl"` (same LoRA settings, completion-only loss).

In [ ]:
import json, os, sys
from collections import Counter
from pathlib import Path

# Run from the repo root so relative paths match the README runbook.
ROOT = Path.cwd()
while not (ROOT / "core").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
PY = sys.executable  # `!{PY} -m ...` runs the CLI with this kernel's interpreter
print("repo root:", ROOT)

In [ ]:
ARMS = ["A1", "A2"]
SEEDS = [0, 1, 2]
BACKEND = "aligntune"  # or "trl"
HUB_ORG = None  # e.g. "my-org" to push checkpoints and adapters
HUB_ARG = f"--hub-org {HUB_ORG}" if HUB_ORG else ""

In [ ]:
for arm in ARMS:
    for seed in SEEDS:
        !{PY} -m adapters.aligntune_sft --arm {arm} --seed {seed} --backend {BACKEND} {HUB_ARG}

## Summary

In [ ]:
for rec in sorted(Path("checkpoints").glob("A[12]-s*/dhanground.json")):
    print(rec.parent.name, json.loads(rec.read_text()))